In [ ]:
import matplotlib.pyplot as plt
from brian2 import *
import pandas as pd
import numpy as np
import networkx as nx
from brian2 import set_device, device

In [ ]:
########################## USEFUL FUNCTIONS #########################
def visualise_connectivity(S):
    Ns = len(S.source)
    Nt = len(S.target)
    figure(figsize=(10, 4))
    subplot(121)
    plot(zeros(Ns), arange(Ns), 'ok', ms=10)
    plot(ones(Nt), arange(Nt), 'ok', ms=10)
    for i, j in zip(S.i, S.j):
        plot([0, 1], [i, j], '-k')
    xticks([0, 1], ['Source', 'Target'])
    ylabel('Neuron index')
    xlim(-0.1, 1.1)
    ylim(-1, max(Ns, Nt))
    subplot(122)
    plot(S.i, S.j, 'ok')
    xlim(-1, Ns)
    ylim(-1, Nt)
    xlabel('Source neuron index')
    ylabel('Target neuron index')
    
def n_inf(v):
    return 1.0 / (1.0 + np.exp((-19.0 - (v)) / 18.0))

In [ ]:
######################### MODEL EQUATIONS #########################  
neuron_eqs = '''
# Ion concentrations
K_i = K_i0 + DK_i : 1  # Potassium concentration in the neuron
Na_i = Na_i0 - DK_i : 1 # Sodium concentration in the neuron
K_o_n = -beta*DK_i : 1 # Potassium concentration variation in the ECS due to the neuron
Na_o = Na_o0 + beta * DK_i : 1 # Sodium concentration in the ECS

# Conductace Variables
n_inf = 1.0/(1.0+exp((-19.0-(V))/18.0)) : 1  # Potassium Conductance Gating Variable
m_inf = 1.0/(1.0+exp((-24.0-(V))/12.0)) : 1  # Sodium Conductance Gating Variable
h = 1.1 - (1.0/(1.0+exp((-8.0)*(n-0.4)))) : 1  # Sodium Inactivation Variable

# Current equations
I_Cl = g_Cl*(V+26.64*log(Cl_o0/Cl_i0)) : 1  # Chloride Current
I_Na = (g_Nal+g_Na*m_inf*h)*(V-26.64*log(Na_o/Na_i)) : 1  # Sodium Current
I_K = (g_Kl + g_K*n)*(V-26.64*log(K_o/K_i)) : 1  # Potassium Current
I_pump = rho/((1.0+exp((21.0-(Na_i))/2.0))*(1.0+exp(5.5-(K_o)))) : 1 # Sodium-Potassium Pump Current	
# I_syn = J*s*(E_syn - V) : 1  # Synaptic Current
I_syn = GsynE*(E_syn_ex - V) + GsynI*(E_syn_inh - V) : 1 # Synaptic Current

# ODE Equations for the neuron model
dV/dt = (-1.0/Cm)*(I_Cl + I_Na + I_K + I_pump - I_syn)/ms : 1 # Membrane voltage
dn/dt = (n_inf-n)/(tau_n*ms) : 1 # Potassium Conductance Gating Variable
dDK_i/dt = -(gamma/w_i)*(I_K - 2.0*I_pump)/ms : 1 # Potassium Concentration in the neuron

# ODE for synaptic conductances
dGsynE/dt = -GsynE/(tau_syn*ms) : 1 # Excitatory synaptic conductance
dGsynI/dt = -GsynI/(tau_syn*ms) : 1 # Inhibitory synaptic conductance
# ds/dt = -s/(tau_syn*ms) : 1 # Synaptic variable

# Neuron position in space
x : metre (constant)
y : metre (constant)
jit_x: metre (constant)
jit_y: metre (constant)

# Parameters Units
Cm : 1 (constant)  # Membrane capacitance
tau_n : 1 (constant)  # Time constant for the gating variable
gamma : 1 (constant)  # Potassium concentration variation factor
w_i : 1 (constant)  # Intracellular volume
w_o : 1 (constant)  # Extracellular volume
beta : 1 (constant) # Ratio of intracellular to extracellular volume
K_bath : 1(constant)  # Potassium concentration in the EB
Na_i0 : 1 (constant)  # Initial Sodium concentration in the neuron
Na_o0 : 1 (constant)  # Initial Sodium concentration in the ECS
K_i0 : 1 (constant)  # Initial Potassium concentration in the neuron
Cl_i0 : 1 (constant)  # Initial Chloride concentration in the neuron
Cl_o0 : 1 (constant)  # Initial Chloride concentration in the ECS
g_Cl : 1 (constant)  # Chloride conductance
g_Na : 1 (constant)  # Sodium conductance
g_K : 1 (constant)  # Potassium conductance
g_Nal : 1 (constant)  # Leak Sodium conductance
g_Kl : 1 (constant)  # Leak Potassium conductance
rho : 1 (constant)  # Sodium-Potassium pump current density
# J : 1 (constant)  # Synaptic conductance excitatory
# E_syn : 1 (constant)  # Synaptic reversal potential excitatory
tau_syn : 1 (constant)  # Synaptic time constant
E_syn_ex : 1 (constant)  # Synaptic reversal potential excitatory
E_syn_inh : 1 (constant)  # Synaptic reversal potential inhibitory
K_o : 1 (linked) # Potassium concentration in the ECS
'''

astrocyte_eqs = '''
# ODE equations for the astrocyte model
dV_a/dt = (-1.0/Cm_a)*(I_Kir + I_K_a + I_l_a + I_K_gap_syn + I_K_gap)/ms : 1  # Astrocyte membrane voltage
# dK_a_t/dt = -(gamma_t/omega_a)*(I_Kir + I_K_a)/ms : 1  # Astrocyte Potassium concentration exchange with the ECS
# dK_a_s/dt = -(gamma_s/omega_a)*S*(I_K_gap_syn + I_K_gap)/ms : 1  # Astrocyte Potassium concentration exchange with the Astrocyte Network/Syncitium
dK_a_t/dt = -gamma_t*(Sigma_a/(omega_a*F))*(I_Kir + I_K_a)/ms : 1  # Astrocyte Potassium concentration exchange with the ECS
dK_a_s/dt = -gamma_s*(Sigma_a/(omega_a*F))*S*(I_K_gap_syn + I_K_gap)/ms : 1  # Astrocyte Potassium concentration exchange with the Astrocyte Network/Syncitium

# Astrocyte current equations
I_Kir = A*g_kir*sqrt(K_o)*((V_a - E_K_a)/(1.0 + exp((V_a - E_K_a)/19.2))) : 1  # Kir current
I_K_a = g_k_a*(V_a - E_K_a) : 1  # Astrocyte Potassium current
I_l_a = g_l_a*(V_a - E_l_a) : 1  # Leak current
I_K_gap_syn = P_K_gap*F*Phi_a*((-K_a0 + (K_a*exp(-Phi_a)))/(-1 + exp(-Phi_a))) : 1  # Gap junction current

# Astrocyte variables
E_K_a = 25.7*log(K_o/K_a) : 1  # Potassium equilibrium potential
Phi_a = 0.04*(V_a - Va_s) : 1  # Astrocyte potential difference
P_K_gap = d_gap*P_K : 1  # Gap junction permeability
S = DK_a/(1.0 + exp((Z_th - K_a_t)*Z_s)) : 1  # Astrocyte variable for gap junctions (Sigmoid)
DK_a = K_a_t + K_a_s : 1 # Variation of Potassium concentration in the astrocyte
K_a = K_a0 + DK_a : 1  # Total Potassium concentration in the astrocyte
Delta_K_o_a = -(omega_a/omega_o) * K_a_t : 1 # Contribution to external K+

# Astrocyte position in space
x_a : metre (constant)
y_a : metre (constant)  
jit_x_a: metre (constant)
jit_y_a: metre (constant)

# Astrocyte parameters
Cm_a : 1 (constant)  # Astrocyte membrane capacitance
g_kir : 1 (constant)  # Maximal Potassium conductance of Kir channel
A : 1 (constant) # Pseudo-Parameter
g_k_a : 1 (constant)  # Potassium Astrocyte conductance
g_l_a : 1 (constant)  # Leak conductance
Sigma_a : 1 (constant)  # Astrocyte surface area
omega_a : 1 (constant)  # Astrocyte intracellular volume
omega_o : 1 (constant)  # Extracellular volume
gamma_t : 1 (constant)  # Conversion factor for astrocyte potassium concentration exchange with the ECS
gamma_s : 1 (constant)  # Conversion factor for astrocyte potassium concentration exchange with the Astrocyte Network/Syncitium
Va_s : 1 (constant)  # Potential of the astrocyte syncitium
E_l_a : 1 (constant)  # Reversal potential of the leak current
P_K : 1 (constant)  # Permeability of the potassium channel
d_gap : 1 (constant)  # Gap junction coupling strength
Z_th : 1 (constant)  # Threshold for the sigmoid function
Z_s : 1 (constant)  # Slope of the sigmoid function
F : 1 (constant)  # Faraday's constant
K_a0 : 1 (constant)  # Initial Potassium concentration in the astrocyte
K_o : 1 (linked) # Potassium concentration in the ECS
I_K_gap : 1 (linked) # Gap junction current
'''

astrocyte_gap_eqs = '''
I_K_gap_1 = g_gap * (V_a_1 - V_a_2) : 1
I_K_gap_2 = g_gap * (V_a_2 - V_a_1) : 1
g_gap : 1 (constant)
V_a_1 : 1 (linked)
V_a_2 : 1 (linked)
'''

external_potassium_eqs = '''
# ODE equation for the external/bath potassium concentration
dK_g/dt = epsilon*(K_bath - K_o)/ms : 1 # Potassium Concentration in the EB
dK_o_modulus/dt = epsilon_modulus*DK_mod/ms : 1 # Potassium concentration in the ECS due to neurons in the other modulus

# External Potassium concentration in the ECS
K_o = K_o0 + K_o_neuron + K_g + K_o_astro + K_o_modulus : 1 # Potassium concentration in the ECS

# External/Bath Potassium parameters
K_o0 : 1 (constant)  # Initial Potassium concentration in the ECS
K_o_neuron : 1 # Potassium concentration in the ECS due to neurons
K_o_astro : 1 (linked) # Potassium concentration in the ECS due to astrocytes
K_bath : 1 (constant)  # Potassium concentration in the EB
epsilon : 1 (constant)  # Potassium concentration in the EB time constant
DK_mod : 1 (linked) # Potassium concentration variation between two modulus
epsilon_modulus : 1 (constant) # Parameter to couple different modules
'''

external_potassium_gap_eqs = '''
DK_mod_1 = K_o_1 - K_o_2 : 1 # Potassium concentration in the ECS due to neurons in the other modulus
DK_mod_2 = K_o_2 - K_o_1 : 1 # Potassium concentration in the ECS due to neurons in the other modulus
K_o_1 : 1 (linked) # Potassium concentration in the ECS of the pre-synaptic module
K_o_2 : 1 (linked) # Potassium concentration in the ECS of the post-synaptic module
'''

In [ ]:
########################## CREATE NEURAL NETWORK ##########################
# Network parameters
N_tot = 10 # Number of neurons
N_modulus = N_tot // 2 # Number of neurons per module
N_e = 4 # Number of excitatory neurons
N_i = N_modulus - N_e # Number of inhibitory neurons

# Create a random graph for the network
p_link = 1  # Probability of edge creation
G = nx.erdos_renyi_graph(int(N_tot), p_link, seed=42)  # Create a random graph
Adj = nx.to_numpy_array(G)  # Adjacency matrix of the graph
Adj = Adj*(1 - np.eye(N_tot))  # Remove self-connections

# Plot the connectivity matrix of Neurons
fig, ax = plt.subplots(figsize=(8, 8))
cax = ax.imshow(Adj, cmap='Greys', interpolation='none', origin='upper')
ax.set_title('Connectivity Matrix')
ax.set_xlabel('Target neuron index')
ax.set_ylabel('Source neuron index')
# Add colorbar with label
cbar = fig.colorbar(cax, ax=ax)
cbar.set_label('Connection presence (1) or absence (0)')
ax.set_xticks(range(Adj.shape[1]))
ax.set_yticks(range(Adj.shape[0]))
ax.grid(which='both', color='gray', linestyle='-', linewidth=0.5)
ax.set_xticklabels(range(Adj.shape[1]))
ax.set_yticklabels(range(Adj.shape[0]))
plt.show()

In [ ]:
################### SETTING INITIAL VALUES ###################
initial_values_ranges = {
        # Neurons
        'V0'     : [-85., -65.],
        'DK_i0'  : [-3., 0.2],
        # Astrocytes
        'V_a0'   : [-95., -75.],
        'K_a_t0' : [-3., 0.5],
        'K_a_s0' : [-2., 2.],
        # Bath
        'K_g0'   : [-0.3, 2.],
    }

# Separate keys by cell type (optional for clarity)
neuron_keys = ['V0', 'DK_i0']
astrocyte_keys = ['V_a0', 'K_a_t0', 'K_a_s0']
bath_keys = ['K_g0']

# ----- Random sampling -----
np.random.seed(42)
initial_values = {}  # Start with an empty dictionary for initial values

# Neurons
for name in neuron_keys:
    low, high = initial_values_ranges[name]
    initial_values[name] = low + (high - low) * np.random.rand(N_tot)  # keeps units

# Astrocytes
for name in astrocyte_keys:
    low, high = initial_values_ranges[name]
    initial_values[name] = low + (high - low) * np.random.rand(2)  # keeps units

# Bath
for name in bath_keys:
    low, high = initial_values_ranges[name]
    initial_values[name] = low + (high - low) * np.random.rand(2)  # keeps units
    
# Extract Initial Parameters
V_init = initial_values.get('V0', -80.)
DK_i_init = initial_values.get('DK_i0', 0.)
V_a_init = initial_values.get('V_a0', -89.)
K_a_t_init = initial_values.get('K_a_t0', 0.)
K_a_s_init = initial_values.get('K_a_s0', 0.)
K_g_init = initial_values.get('K_g0', 0.)

In [ ]:
######################### MODEL SIMULATION ########################
device.reinit()
device.activate()
set_device('cpp_standalone', build_on_run=False)
seed(42)  # to get identical figures for repeated runs
# Set Brian2 preferences
prefs.codegen.target = 'numpy'
prefs.codegen.loop_invariant_optimisations = True

net = Network()

# Parameters for simulation
defaultclock.dt = 0.03*msecond
t_sim = 90*second
K_bath_temp = 11.
Z_th_temp = 100.
g_gap_temp = 5

###############################################################
######################## Neural Network #######################
###############################################################
neurons_1 = NeuronGroup(N_modulus, neuron_eqs, threshold='V > 10', refractory='V > 10', reset='',
                method='rk4')
neurons_2 = NeuronGroup(N_modulus, neuron_eqs, threshold='V > 10', refractory='V > 10', reset='',
                method='rk4')

np.random.seed(42)
# Neuron Initial Conditions
neurons_1.V = V_init[:N_modulus]  # Initial membrane potential
neurons_1.n = [n_inf(temp) for temp in V_init[:N_modulus]]# Initial Potassium conductance gating variable
neurons_1.DK_i = DK_i_init[:N_modulus] # Initial Potassium concentration variation in the neuron

neurons_2.V = V_init[N_modulus:]  # Initial membrane potential
neurons_2.n = [n_inf(temp) for temp in V_init[N_modulus:]]# Initial Potassium conductance gating variable
neurons_2.DK_i = DK_i_init[N_modulus:] # Initial Potassium concentration variation in the neuron

for idx, neurons in enumerate([neurons_1, neurons_2]):
    # Neuron parameters
    neurons.Cm = 40.
    neurons.tau_n = 10.
    neurons.g_Cl = 7.5
    neurons.g_Na = 40.0
    neurons.g_K = 22.0
    neurons.g_Nal = 0.02
    neurons.g_Kl = 0.12
    neurons.w_i = 2160.0
    neurons.gamma = 0.002
    neurons.rho = 250.0
    neurons.Na_i0 = 16.0
    neurons.K_i0 = 140.0
    neurons.Cl_i0 = 5.0

    # External / ECS & Bath parameters
    neurons.Na_o0 = 138.0
    neurons.Cl_o0 = 112.0
    neurons.w_o = 720.0
    neurons.beta = neurons.w_i / neurons.w_o  # Ratio of intracellular to extracellular volume

    # Synaptic parameters
    neurons.GsynE = 0.  # Initial Excitatory synaptic conductance
    neurons.GsynI = 0.  # Initial Inhibitory synaptic conductance
    neurons.E_syn_ex = 0.0  # Synaptic reversal potential excitatory
    neurons.E_syn_inh = -80.0  # Synaptic reversal potential inhibitory
    neurons.tau_syn = 5.0  # Synaptic time constant
    # neurons.E_syn = 0.0  # Synaptic reversal potential
    # neurons.J = 0.1  # Synaptic conductance
    # neurons.s = 0.0  # Synaptic variable
    
###############################################################
########################### Synapses ##########################
###############################################################
# Create synapses based on the connectivity matrix A
# Extract sources and targets from connectivity matrix
sources_exc_11, targets_exc_11 = Adj[:N_e,:N_modulus].nonzero()  # Excitatory neurons for module 1
sources_inh_11, targets_inh_11 = Adj[N_e:N_modulus,:N_modulus].nonzero()  # Inhibitory neurons for module 1
sources_inh_11 += N_e

sources_exc_22, targets_exc_22 = Adj[N_modulus:N_modulus+N_e,N_modulus:].nonzero()  # Excitatory neurons for module 2
sources_inh_22, targets_inh_22 = Adj[N_modulus+N_e:,N_modulus:].nonzero()  # Inhibitory neurons for module 2
sources_inh_22 += N_e

sources_exc_12, targets_exc_12 = Adj[:N_e,N_modulus:].nonzero()  # Excitatory neurons from module 1 to module 2
sources_inh_12, targets_inh_12 = Adj[N_e:N_modulus,N_modulus:].nonzero()  # Inhibitory neurons from module 1 to module 2
sources_inh_12 += N_e

sources_exc_21, targets_exc_21 = Adj[N_modulus:N_modulus+N_e,:N_modulus].nonzero()  # Excitatory neurons from module 2 to module 1
sources_inh_21, targets_inh_21 = Adj[N_modulus+N_e:,:N_modulus].nonzero()  # Inhibitory neurons from module 2 to module 1
sources_inh_21 += N_e

# Synapses update target neuron's S variable on presynaptic spike (see Depannemaecker et al. 2022, https://doi.org/10.1523/ENEURO.0234-21.2022)
np.random.seed(42)

Q_e_range = [0.5, 0.7]
Q_i_range = [1, 3]

Qi_11 = Q_i_range[0] + (Q_i_range[1] - Q_i_range[0]) * np.random.rand(1)
Qe_11 = Q_e_range[0] + (Q_e_range[1] - Q_e_range[0]) * np.random.rand(4)

Qi_22 = Q_i_range[0] + (Q_i_range[1] - Q_i_range[0]) * np.random.rand(1)
Qe_22 = Q_e_range[0] + (Q_e_range[1] - Q_e_range[0]) * np.random.rand(4)

Qi_12 = Q_i_range[0] + (Q_i_range[1] - Q_i_range[0]) * np.random.rand(1)
Qe_12 = Q_e_range[0] + (Q_e_range[1] - Q_e_range[0]) * np.random.rand(4)

Qi_21 = Q_i_range[0] + (Q_i_range[1] - Q_i_range[0]) * np.random.rand(1)
Qe_21 = Q_e_range[0] + (Q_e_range[1] - Q_e_range[0]) * np.random.rand(4)

S_exc_11 = Synapses(neurons_1, neurons_1, model='Qe : 1', on_pre='GsynE_post += Qe') # Excitatory synapse in module 1
S_exc_11.connect(i=sources_exc_11, j=targets_exc_11)
S_exc_11.Qe = np.repeat(Qe_11, len(sources_exc_11)//N_e)

S_inh_11 = Synapses(neurons_1, neurons_1, model='Qi : 1', on_pre='GsynI_post += Qi') # Inhibitory synapse in module 1
S_inh_11.connect(i=sources_inh_11, j=targets_inh_11)
S_inh_11.Qi = Qi_11

S_exc_22 = Synapses(neurons_2, neurons_2, model='Qe : 1', on_pre='GsynE_post += Qe') # Excitatory synapse in module 2
S_exc_22.connect(i=sources_exc_22, j=targets_exc_22)
S_exc_22.Qe = np.repeat(Qe_22, len(sources_exc_22)//N_e)

S_inh_22 = Synapses(neurons_2, neurons_2, model='Qi : 1', on_pre='GsynI_post += Qi') # Inhibitory synapse in module 2
S_inh_22.connect(i=sources_inh_22, j=targets_inh_22)
S_inh_22.Qi = Qi_22

S_exc_12 = Synapses(neurons_1, neurons_2, model='Qe : 1', on_pre='GsynE_post += Qe') # Excitatory synapse from module 1 to module 2
S_exc_12.connect(i=sources_exc_12, j=targets_exc_12)
S_exc_12.Qe = np.repeat(Qe_12, len(sources_exc_12)//N_e)

S_inh_12 = Synapses(neurons_1, neurons_2, model='Qi : 1', on_pre='GsynI_post += Qi') # Inhibitory synapse from module 1 to module 2
S_inh_12.connect(i=sources_inh_12, j=targets_inh_12)
S_inh_12.Qi = Qi_12

S_exc_21 = Synapses(neurons_2, neurons_1, model='Qe : 1', on_pre='GsynE_post += Qe') # Excitatory synapse from module 2 to module 1
S_exc_21.connect(i=sources_exc_21, j=targets_exc_21)
S_exc_21.Qe = np.repeat(Qe_21, len(sources_exc_21)//N_e)

S_inh_21 = Synapses(neurons_2, neurons_1, model='Qi : 1', on_pre='GsynI_post += Qi') # Inhibitory synapse from module 2 to module 1
S_inh_21.connect(i=sources_inh_21, j=targets_inh_21)
S_inh_21.Qi = Qi_21

###############################################################
########################## Astrocyte ##########################
###############################################################
# Astrocyte Group
astrocytes_1 = NeuronGroup(1, astrocyte_eqs, method='rk4')
astrocytes_2 = NeuronGroup(1, astrocyte_eqs, method='rk4')

# Astrocyte Initial Conditions
astrocytes_1.V_a = V_a_init[0] # Initial membrane potential
astrocytes_1.K_a_t = K_a_t_init[0]  # Initial Potassium concentration between astrocytes and ECS
astrocytes_1.K_a_s = K_a_s_init[0]  # Initial Potassium concentration in the astrocyte syncitium
astrocytes_1.d_gap = 0  # Gap junction coupling strength

astrocytes_2.V_a = V_a_init[1] # Initial membrane potential
astrocytes_2.K_a_t = K_a_t_init[1]  # Initial Potassium concentration between astrocytes and ECS
astrocytes_2.K_a_s = K_a_s_init[1]  # Initial Potassium concentration in the astrocyte syncitium
astrocytes_2.d_gap = 1  # Gap junction coupling strength

for astrocytes in [astrocytes_1, astrocytes_2]:
    # Astrocyte parameters
    astrocytes.Cm_a = 100.0  # Membrane capacitance
    astrocytes.g_kir = 10.0  # Maximal Potassium conductance
    astrocytes.A = 1.0  # Pseudo-Parameter
    astrocytes.g_k_a = 0.0  # Potassium Astrocyte conductance
    astrocytes.g_l_a = 0.1  # Leak conductance
    astrocytes.Sigma_a = 1600.0  # Astrocyte surface area
    astrocytes.omega_a = 2000.0  # Astrocyte intracellular volume
    astrocytes.omega_o = 720.0  # Extracellular volume
    astrocytes.gamma_t = 5.0  # Conversion factor for astrocyte potassium concentration exchange with the ECS
    astrocytes.gamma_s = 6.0  # Conversion factor for astrocyte potassium concentration exchange with the Astrocyte Network/Syncitium
    astrocytes.Va_s = -90.0  # Potential of the astrocyte syncitium
    astrocytes.E_l_a = -70.0  # Reversal potential of the leak current
    astrocytes.P_K = 6e-5  # Permeability of the potassium channel
    astrocytes.Z_th = Z_th_temp  # Threshold for the sigmoid function
    astrocytes.Z_s = 0.05  # Slope of the sigmoid function
    astrocytes.F = 96458.0  # Faraday's constant
    astrocytes.K_a0 = 135.0  # Initial Potassium concentration in the astrocyte

astrocyte_gap = NeuronGroup(1, astrocyte_gap_eqs, method='rk4')
astrocyte_gap.V_a_1 = linked_var(astrocytes_1, 'V_a')  # Astrocyte membrane voltage of the pre-synaptic astrocyte
astrocyte_gap.V_a_2 = linked_var(astrocytes_2, 'V_a')  # Astrocyte membrane voltage of the post-synaptic astrocyte 
astrocyte_gap.g_gap = g_gap_temp  # Gap junction conductance

astrocytes_1.I_K_gap = linked_var(astrocyte_gap, 'I_K_gap_1')  # Gap junction current
astrocytes_2.I_K_gap = linked_var(astrocyte_gap, 'I_K_gap_2')  # Gap junction current

###############################################################
######################## Link Astro-SNN #######################
###############################################################
external_potassium_1 = NeuronGroup(1, external_potassium_eqs, method='rk4')
external_potassium_2 = NeuronGroup(1, external_potassium_eqs, method='rk4')

# Initial Conditions External Potassium
external_potassium_1.K_g = K_g_init[0]  # Initial Potassium concentration in the ECS due to EB
    
external_potassium_2.K_g = K_g_init[1]  # Initial Potassium concentration in the ECS due to EB

for external_potassium, neurons, astrocytes in zip([external_potassium_1, external_potassium_2], [neurons_1, neurons_2], [astrocytes_1, astrocytes_2]):
    external_potassium.K_o0 = 4.8  # Initial Potassium concentration in the ECS
    external_potassium.K_bath = K_bath_temp  # Potassium concentration in the EB
    external_potassium.epsilon = 0.01  # Potassium concentration in the EB time constant
    external_potassium.K_o_neuron = 0.0  # Initial Potassium concentration in the ECS due to neurons
    external_potassium.K_o_modulus = 0.0  # Initial Potassium concentration in the ECS due to neurons in the other modulus
    external_potassium.epsilon_modulus = 0.001 # Parameter to couple different modules

    external_potassium.K_o_astro = linked_var(astrocytes, 'Delta_K_o_a')  # Potassium concentration in the ECS due to the astrocyte

    # Link to the neurons' and astrocytes' external potassium concentration
    neurons.K_o = linked_var(external_potassium, 'K_o')  # Potassium concentration in the ECS due to the astrocyte
    astrocytes.K_o = linked_var(external_potassium, 'K_o')  # Potassium concentration in the ECS

neuron_Ko_synapses_1 = Synapses(neurons_1, external_potassium_1, 
                                model= '''
                                K_o_neuron_post = K_o_n_pre : 1 (summed)
                                ''', on_pre='')  # No direct synaptic effect

neuron_Ko_synapses_1.connect()  # Connect all neurons to each other to share K_o_n

neuron_Ko_synapses_2 = Synapses(neurons_2, external_potassium_2, 
                                model= '''
                                K_o_neuron_post = K_o_n_pre : 1 (summed)
                                ''', on_pre='')  # No direct synaptic effect
neuron_Ko_synapses_2.connect()  # Connect all neurons to each other to share K_o_n

###############################################################
######################## Link 2 Modules #######################
###############################################################
external_potassium_eqs_gap = NeuronGroup(1, external_potassium_gap_eqs, method='rk4')
external_potassium_eqs_gap.K_o_1 = linked_var(external_potassium_1, 'K_o')  # Potassium concentration in the ECS of the pre-synaptic module
external_potassium_eqs_gap.K_o_2 = linked_var(external_potassium_2, 'K_o')  # Potassium concentration in the ECS of the post-synaptic module

external_potassium_1.DK_mod = linked_var(external_potassium_eqs_gap, 'DK_mod_1')  # Potassium concentration variation between two modulus
external_potassium_2.DK_mod = linked_var(external_potassium_eqs_gap, 'DK_mod_2')  # Potassium concentration variation between two modulus

###############################################################
######################## Run Simulation #######################
###############################################################
# Define Spike and State Monitors
spikemon_1 = SpikeMonitor(neurons_1)
neurons_mon_1 = StateMonitor(neurons_1, ['V', 'DK_i', 'n', 'K_o', 'I_syn', 'GsynE', 'GsynI'], record=True)
astrocytes_mon_1 = StateMonitor(astrocytes_1, ['V_a', 'K_a_t', 'K_a_s', 'K_o', 'I_K_gap', 'I_Kir'], record=True)
external_potassium_mon_1 = StateMonitor(external_potassium_1, ['K_o', 'K_g'], record=True)

spikemon_2 = SpikeMonitor(neurons_2)
neurons_mon_2 = StateMonitor(neurons_2, ['V', 'DK_i', 'n', 'K_o', 'I_syn', 'GsynE', 'GsynI'], record=True)
astrocytes_mon_2 = StateMonitor(astrocytes_2, ['V_a', 'K_a_t', 'K_a_s', 'K_o', 'I_K_gap', 'I_Kir'], record=True)
external_potassium_mon_2 = StateMonitor(external_potassium_2, ['K_o', 'K_g'], record=True)

# Define Network object
net.add(neurons_1)
net.add(neurons_2)
net.add(S_exc_11)
net.add(S_inh_11)
net.add(S_exc_22)
net.add(S_inh_22)
net.add(S_exc_12)
net.add(S_exc_21)
net.add(S_inh_12)
net.add(S_inh_21)
net.add(astrocytes_1)
net.add(astrocytes_2)
net.add(astrocyte_gap)
net.add(external_potassium_1)
net.add(external_potassium_2) 
net.add(neuron_Ko_synapses_1)
net.add(neuron_Ko_synapses_2)
net.add(external_potassium_eqs_gap)
net.add(spikemon_1)
net.add(neurons_mon_1)
net.add(astrocytes_mon_1)
net.add(external_potassium_mon_1)
net.add(spikemon_2)
net.add(neurons_mon_2)
net.add(astrocytes_mon_2)
net.add(external_potassium_mon_2)

# Run the simulation
net.run(t_sim, report='text')

device.build(run=True)   # explicitly build and run the compiled code

In [ ]:
######################### Plot Results ########################
for idx, (spikemon, neurons_mon, astrocytes_mon, external_potassium_mon) in enumerate(zip([spikemon_1, spikemon_2], [neurons_mon_1, neurons_mon_2], [astrocytes_mon_1, astrocytes_mon_2], [external_potassium_mon_1, external_potassium_mon_2])):

    n_mod = idx + 1
    
    font_size_title = 15
    
    N_idx = 0
    
    # Neurons Plots    
    fig, ax = plt.subplots(4, 2, figsize=(18, 15), gridspec_kw={'height_ratios':[1,1,1,1]})
    fig.suptitle(f'Neurons Plots in Module {n_mod}', fontsize=18, fontweight='bold')

    # Make the first plot span both columns by using gridspec directly
    gs = ax[0,0].get_gridspec()

    # Remove the first row axes
    for a in ax[0,:]:
        a.remove()

    # Create new axes spanning both columns in first row
    ax0 = fig.add_subplot(gs[0, :])

    # Now plot on ax0
    ax0.plot(spikemon.t/msecond, spikemon.i, '.')
    ax0.set_xlim(0, neurons_mon.t[-1]/msecond)
    ax0.set_xlabel('Time (ms)')
    ax0.set_ylabel('Neuron index')
    ax0.set_title('Spike raster plot', fontsize=font_size_title, fontweight='bold')
        
    ax[1, 0].plot(neurons_mon.t/msecond, neurons_mon.V[N_idx])
    ax[1, 0].set_xlabel('Time (ms)')
    ax[1, 0].set_ylabel('Membrane potential (mV)')
    ax[1, 0].set_title(f'Membrane Potential for Neuron {N_idx} (V)', fontsize=font_size_title, fontweight='bold')

    ax[1, 1].plot(neurons_mon.t/msecond, neurons_mon.DK_i[N_idx])
    ax[1, 1].set_xlabel('Time (ms)')
    ax[1, 1].set_ylabel('Potassium concentration (mmolar)')
    ax[1, 1].set_title(f'Internal Potassium Concentration Variation for Neuron {N_idx}' + r' ($\Delta K_i$)', fontsize=font_size_title, fontweight='bold')

    ax[2, 0].plot(neurons_mon.t/msecond, neurons_mon.n[N_idx])
    ax[2, 0].set_xlabel('Time (ms)')
    ax[2, 0].set_ylabel('Potassium conductance gating variable (unitless)')
    ax[2, 0].set_title(f'Potassium Conductance Gating Variable for Neuron {N_idx} (n)', fontsize=font_size_title, fontweight='bold')

    ax[2, 1].plot(neurons_mon.t/msecond, neurons_mon.K_o[N_idx])
    ax[2, 1].set_xlabel('Time (ms)')
    ax[2, 1].set_ylabel('Potassium concentration (mmolar)')
    ax[2, 1].set_title(f'Potassium Concentration in the ECS for Neuron {N_idx}' + r' ($K_o$)', fontsize=font_size_title, fontweight='bold')
    
    ax[3, 0].plot(neurons_mon.t/ms, neurons_mon.I_syn[N_idx])
    ax[3, 0].set_xlabel('Time (ms)')
    ax[3, 0].set_ylabel('Synaptic current (pA)')
    ax[3, 0].set_title(f'Synaptic Current for Neuron {N_idx}', fontsize=font_size_title, fontweight='bold')

    ax[3, 1].plot(external_potassium_mon.t/msecond, external_potassium_mon.K_g[0])
    ax[3, 1].set_xlabel('Time (ms)')
    ax[3, 1].set_ylabel('Potassium concentration (mmolar)')
    ax[3, 1].set_title('Potassium Concentration in the EB' + r' ($K_g$)', fontsize=font_size_title, fontweight='bold')

    plt.tight_layout(rect=[0, 0.03, 1, 0.92])
    fig.subplots_adjust(top=0.93)
    plt.show(fig)
    
    # Plotting Neuron Synaptic Conductances
    fig, ax = plt.subplots(3,1,figsize=(18, 8))
    fig.suptitle(f'Neuron Synaptic Variables of Neuron {N_idx} in Module {n_mod}')
    ax[0].plot(neurons_mon.t/msecond, neurons_mon.I_syn[N_idx])
    ax[0].set_xlabel('Time (ms)')
    ax[0].set_ylabel('Synaptic current (pA)')
    ax[0].set_title('Synaptic Current ' + r'($I_{syn}$)') 
    
    ax[1].plot(neurons_mon.t/msecond, neurons_mon.GsynE[N_idx])
    ax[1].set_xlabel('Time (ms)')
    ax[1].set_ylabel('Conductance (nS)')
    ax[1].set_title('Excitatory Synaptic Conductance' + r' ($G_{E,syn}$)') 

    ax[2].plot(neurons_mon.t/msecond, neurons_mon.GsynI[N_idx])
    ax[2].set_xlim([4600, 4700])
    ax[2].set_xlabel('Time (ms)')
    ax[2].set_ylabel('Conductance (nS)')
    ax[2].set_title('Inhibitory Synaptic Conductance' + r' ($G_{I,syn}$)') 

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to fit suptitle
    plt.show(fig)

    # Plotting State Variables Astrocyte 0
    fig, ax = plt.subplots(3,2,figsize=(18, 10), gridspec_kw={'height_ratios':[1,1,1]})
    fig.suptitle(f'State Variables of Astrocyte in Module {n_mod}', fontsize=18, fontweight='bold')
    
    ax[0, 0].plot(astrocytes_mon.t/msecond, astrocytes_mon.V_a[0])
    ax[0, 0].set_xlabel('Time (ms)')
    ax[0, 0].set_ylabel('Membrane potential (mV)')
    ax[0, 0].set_title('Membrane Potential (V)', fontsize=font_size_title, fontweight='bold')

    ax[0, 1].plot(astrocytes_mon.t/msecond, astrocytes_mon.K_a_t[0])
    ax[0, 1].set_xlabel('Time (ms)')
    ax[0, 1].set_ylabel('Potassium concentration (mmolar)')
    ax[0, 1].set_title('Potassium Exchange with ECS' + r' ($K_{a,t}$)', fontsize=font_size_title, fontweight='bold')

    ax[1, 0].plot(astrocytes_mon.t/msecond, astrocytes_mon.K_a_s[0])
    ax[1, 0].set_xlabel('Time (ms)')
    ax[1, 0].set_ylabel('Potassium concentration (mmolar)')
    ax[1, 0].set_title('Potassium Exchange with Syncitium' + r' ($K_{a,s}$)', fontsize=font_size_title, fontweight='bold')

    ax[1, 1].plot(astrocytes_mon.t/msecond, astrocytes_mon.K_o[0])
    ax[1, 1].set_xlabel('Time (ms)')
    ax[1, 1].set_ylabel('Potassium Concentration (mmolar)')
    ax[1, 1].set_title('Potassium Concentration in the ECS' + r' ($K_o$)', fontsize=font_size_title, fontweight='bold')
    
    ax[2,0].plot(astrocytes_mon.t/msecond, astrocytes_mon.I_Kir[0])
    ax[2,0].set_xlabel('Time (ms)')
    ax[2,0].set_ylabel('Kir Current (pA)')
    ax[2,0].set_title('Kir Current' + r' ($I_{Kir}$)', fontsize=font_size_title, fontweight='bold') 

    ax[2,1].plot(astrocytes_mon.t/msecond, astrocytes_mon.I_K_gap[0])
    ax[2,1].set_xlabel('Time (ms)')
    ax[2,1].set_ylabel('Gap Junction Current (pA)')
    ax[2,1].set_title('Gap Junction Current' + r' ($I_{Kgap}$)', fontsize=font_size_title, fontweight='bold')

    plt.tight_layout(rect=[0, 0.03, 1, 0.92])
    fig.subplots_adjust(top=0.90)
    plt.show(fig)